In [0]:
from pyspark.sql import functions as F

raw_path = "/Volumes/company_risk_intelligence_platform/bronze/raw_data/yfinance/stock"

df = (
    spark.read
    .option("multiline", "true")   # if JSON spans multiple lines
    .json(raw_path)
)

display(df)

In [0]:
spark.read.option("multiline", "true").json(raw_path).printSchema()

In [0]:
from pyspark.sql.functions import col
table_mapping = {
    "yf_stock": "stock",
    "yf_news": "news",
    "yf_income_statement": "income_statement",
    "yf_balance_sheet": "balance_sheet",
    "yf_cashflow": "cashflow",
    "yf_info": "info"
}
df = df.toDF(*[
    c.lower().replace(" ", "_")
    for c in df.columns
])
display(df)

In [0]:
df.write.mode("overwrite").saveAsTable("company_risk_intelligence_platform.bronze.yf_stock")

In [0]:
from pyspark.sql.functions import current_timestamp, col
import re

base_path = "/Volumes/company_risk_intelligence_platform/bronze/raw_data/yfinance/"

table_mapping = {
    "yf_stock": "stock",
    "yf_news": "news",
    "yf_income_statement": "income_statement",
    "yf_balance_sheet": "balance_sheet",
    "yf_cashflow": "cashflow",
    "yf_info": "info"
}

catalog = "company_risk_intelligence_platform"
schema = "bronze"

# -----------------------------------
# Clean column names
# -----------------------------------
def clean_column_names(df):

    cleaned_cols = []

    for c in df.columns:

        clean_name = (
            c.strip() #remove whitespaces
             .lower()
             .replace(" ", "_")
             .replace("-", "_")
        )

        clean_name = re.sub(r"[^a-zA-Z0-9_]", "", clean_name)

        cleaned_cols.append(clean_name)

    return df.toDF(*cleaned_cols)

# -----------------------------------
# Add metadata columns
# -----------------------------------
def add_metadata_columns(df):

    return (
        df.withColumn("ingestion_ts", current_timestamp())
          .withColumn("file_path", col("_metadata.file_path"))
    )

# -----------------------------------
# Main ingestion loop
# -----------------------------------
for table, folder in table_mapping.items():

    print(f"Processing {table}")

    input_path = f"{base_path}{folder}/"

    try:

        # Read JSON
        df = (
            spark.read
                .option("multiline", "true")
                .option("recursiveFileLookup", "true")
                .json(input_path)
        )

        # Apply transformations
        df = clean_column_names(df)

        df = add_metadata_columns(df)

        # Write Delta table
        (
            df.write.format("delta")
              .mode("overwrite")
              .option("overwriteSchema", "true")
              .saveAsTable(f"{catalog}.{schema}.{table}")
        )

        print(f"Loaded {table}")

    except Exception as e:

        print(f"Skipping {table}: {e}")

print("All Yahoo Finance bronze tables processed")